# Stage 09 / Lecture 59: Creating Unified AI Agents with `create_agent` and Tool Calling

This production reference notebook demonstrates the updated **LangChain v1+** `create_agent` architecture:
1. **Model Initialization & Defensive Fallback**: Using `ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)` with graceful offline simulation fallback if credentials are absent.
2. **Graph Topology Inspection**: Contrasting a tool-less linear agent (`START -> model -> END`) with a cyclic tool-enabled agent (`START -> model <-> tools -> END`).
3. **API Guardrails**: Explicitly demonstrating why legacy parameters (`verbose=True`) and raw string inputs fail, and enforcing the `{"messages": [...]}` dictionary contract.
4. **Full Trajectory Inspection**: Auditing `HumanMessage`, `AIMessage` (with `tool_calls`), `ToolMessage`, and the final grounded `AIMessage`.


In [1]:
from typing import List, Dict, Any, Optional
import os
from dotenv import load_dotenv

import langchain
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage

load_dotenv()

groq_key_present = bool(os.getenv("GROQ_API_KEY"))
print(f"LangChain version: {langchain.__version__}")
print(f"GROQ_API_KEY configured: {groq_key_present}")


LangChain version: 1.4.3
GROQ_API_KEY configured: True


### 1. Creating a Tool-Less Agent and Detecting Legacy Parameter Errors

When `tools=[]` is passed to `create_agent`, the compiled graph contains only `model`. Note that legacy `verbose=True` raises a `TypeError` because `create_agent` builds a compiled LangGraph object rather than `AgentExecutor`.


In [2]:
# Initialize LLM per workspace default
if groq_key_present:
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
else:
    raise EnvironmentError("GROQ_API_KEY is required to run live ChatGroq tool-calling verification.")

# 1. Verify legacy `verbose=True` raises TypeError in LangChain v1+
try:
    create_agent(model=llm, tools=[], system_prompt="You are a helpful assistant.", verbose=True)
    raise AssertionError("Expected TypeError for legacy verbose parameter.")
except TypeError as exc:
    print(f"Verified expected TypeError for deprecated parameter: {exc}")

# 2. Construct valid tool-less agent
toolless_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant."
)

toolless_nodes = list(toolless_agent.get_graph().nodes.keys())
assert toolless_nodes == ["__start__", "model", "__end__"], f"Unexpected nodes: {toolless_nodes}"
print(f"Tool-less Agent Graph Nodes: {toolless_nodes}")


Verified expected TypeError for deprecated parameter: create_agent() got an unexpected keyword argument 'verbose'
Tool-less Agent Graph Nodes: ['__start__', 'model', '__end__']


### 2. Binding Typed Functions with Docstrings as Agent Tools

The LLM relies strictly on the Python function's name, type annotations (`city: str -> str`), and **Docstring** to decide when and how to invoke external tools.


In [3]:
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

agent_nodes = list(weather_agent.get_graph().nodes.keys())
assert agent_nodes == ["__start__", "model", "tools", "__end__"], f"Unexpected nodes: {agent_nodes}"
print(f"Tool-Enabled Agent Graph Nodes: {agent_nodes}")


Tool-Enabled Agent Graph Nodes: ['__start__', 'model', 'tools', '__end__']


### 3. Enforcing Invocation Contracts and Inspecting the Execution Trace

`agent.invoke(...)` expects a dictionary containing a `"messages"` key. Passing a raw string raises an exception. However, passing a raw string inside `{"messages": "..."}` or a list of role dictionaries works seamlessly, even with minor user typos.


In [4]:
# 1. Demonstrate raw string input failure
try:
    weather_agent.invoke("What is the weather in New York?")
    raise AssertionError("Expected error when invoking CompiledStateGraph with raw string.")
except Exception as exc:
    print(f"Verified expected schema failure on raw string input: {type(exc).__name__}")

# 2. Standard dictionary invocation
response = weather_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in New York?"}]
})

messages: List[BaseMessage] = response["messages"]
assert len(messages) == 4, f"Expected 4 trajectory messages, got {len(messages)}"
assert isinstance(messages[0], HumanMessage)
assert isinstance(messages[1], AIMessage) and len(messages[1].tool_calls) > 0
assert messages[1].tool_calls[0]["name"] == "get_weather"
assert isinstance(messages[2], ToolMessage)
assert "sunny" in messages[2].content.lower()
assert isinstance(messages[3], AIMessage) and "sunny" in messages[3].content.lower()

print("\nExecution Trajectory Breakdown:")
for idx, msg in enumerate(messages, 1):
    role_name = type(msg).__name__
    detail = msg.content if msg.content else f"tool_calls={msg.tool_calls}"
    print(f"  [{idx}] {role_name}: {detail}")

# 3. Shorthand string inside `messages` key with intentional typo
typo_response = weather_agent.invoke({"messages": "What is the wather in New York?"})
final_answer = typo_response["messages"][-1].content
assert "sunny" in final_answer.lower()
print(f"\nTypo-Resilient Shorthand Answer: {final_answer}")
print("\nAll Stage 09 / Lecture 59 agent and tool-calling assertions passed.")


Verified expected schema failure on raw string input: InvalidUpdateError



Execution Trajectory Breakdown:
  [1] HumanMessage: What is the weather in New York?
  [2] AIMessage: tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'fc_6ed08d52-3887-4436-92e0-b38ec8ddd2f4', 'type': 'tool_call'}]
  [3] ToolMessage: The weather in New York is sunny.
  [4] AIMessage: The weather in New York is sunny.



Typo-Resilient Shorthand Answer: The weather in New York is sunny.

All Stage 09 / Lecture 59 agent and tool-calling assertions passed.
